# NOAA GEFS — Probability Analysis

**Model:** NOAA Global Ensemble Forecast System — 31 members  
**Station:** TGPY — Maurice Bishop International Airport, Grenada

Reads GEFS ensemble mean, spread, and empirical precipitation probability.

In [ ]:
import cfgrib
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

TGPY_LAT =  12.0042
TGPY_LON = -61.7862
UTC_OFFSET = pd.Timedelta(hours=-4)

RUN_DIR = Path("data") / sorted((Path("data")).iterdir())[-1].name

def nearest_point(ds):
    return ds.sel(latitude=TGPY_LAT, longitude=TGPY_LON, method="nearest", tolerance=0.5)

def grib_to_df(filepath):
    datasets = cfgrib.open_datasets(str(filepath))
    series = {}
    for ds in datasets:
        pt = nearest_point(ds)
        for var in pt.data_vars:
            da  = pt[var]
            vt  = da.valid_time.values
            idx = pd.to_datetime([vt]) if vt.ndim == 0 else pd.to_datetime(vt)
            val = [float(da.values)] if vt.ndim == 0 else da.values.astype(float)
            series[var] = pd.Series(val, index=idx)
    return pd.DataFrame(series).sort_index()

print(f"Run: {RUN_DIR.name}")

In [ ]:
# ── GEFS mean T2m and MSL time series ────────────────────────────────────────
df_mean = grib_to_df(RUN_DIR / "gefs_mean_f000-f120_6h.grib2")
df_sprd = grib_to_df(RUN_DIR / "gefs_spread_f000-f120_6h.grib2")

# Normalise GFS column names
for src, dst in [("prmsl", "msl")]:
    for df in [df_mean, df_sprd]:
        if src in df.columns and dst not in df.columns:
            df.rename(columns={src: dst}, inplace=True)

run_init = df_mean.index[0]

print("GEFS Ensemble Mean — T2m and MSL at TGPY (0–120h)")
print("-" * 60)
print(f"{'Step':>5}  {'AST':>14}  {'T2m mean':>9}  {'T2m sprd':>9}  {'MSL mean':>9}")
print("-" * 60)
for ts in df_mean.index[:21]:  # first 120h
    step_h = int((pd.Timestamp(ts) - pd.Timestamp(run_init)).total_seconds() / 3600)
    ast    = (pd.Timestamp(ts) + UTC_OFFSET).strftime("%d%b %H:%M")
    t_mean = float(df_mean["t2m"][ts]) - 273.15 if "t2m" in df_mean.columns else float("nan")
    t_sprd = float(df_sprd["t2m"][ts])           if "t2m" in df_sprd.columns else float("nan")
    m_mean = float(df_mean["msl"][ts]) / 100      if "msl" in df_mean.columns else float("nan")
    print(f"  {step_h:>3}  {ast:>14}  {t_mean:>9.1f}  {t_sprd:>9.2f}  {m_mean:>9.1f}")

In [ ]:
# ── Precipitation probability ─────────────────────────────────────────────────
ds_prob  = xr.open_dataset(RUN_DIR / "gefs_prob_precip_d1-d5.nc")
windows  = ds_prob["window"].values.tolist()
day_labels = ["D1", "D2", "D3", "D4", "D5"]

print("\nGEFS — Precipitation Probability at TGPY (empirical, 31 members)")
print("-" * 50)
print(f"{'Window':>12}  {'P(>1mm/day)':>12}  {'P(>5mm/day)':>12}  {'Signal':>8}")
print("-" * 50)
for i, (w, d) in enumerate(zip(windows, day_labels)):
    p1  = float(ds_prob["tpg1"].values[i])
    p5  = float(ds_prob["tpg5"].values[i])
    sig = "HIGH" if p1 >= 0.7 else ("MOD" if p1 >= 0.4 else "LOW")
    print(f"  {d} {w:>8}  {p1:>12.0%}  {p5:>12.0%}  {sig:>8}")

In [ ]:
# ── Canonical derivation + export ─────────────────────────────────────────────
CANONICAL_COLS = [
    # Core (original 11)
    "t2m_c", "msl_hpa", "wspd_kt", "wdir", "d2m_c", "tcc_pct",
    "tp_mm", "tcwv", "fg10_kt", "sp_hpa", "cape",
    # New tropical params
    "skt_c", "cp_mm", "lcc_pct", "mcc_pct", "hcc_pct",
    "cin", "olr_wm2", "blh_m", "lhf", "shf",
]

gm = df_mean.copy()
# df_mean already has 'msl' (prmsl renamed above); rename 'gust' → 'fg10'
if "gust" in gm.columns and "fg10" not in gm.columns:
    gm = gm.rename(columns={"gust": "fg10"})

if "t2m"   in gm.columns: gm["t2m_c"]   = gm["t2m"] - 273.15
if "msl"   in gm.columns: gm["msl_hpa"] = gm["msl"] / 100
if "sp"    in gm.columns: gm["sp_hpa"]  = gm["sp"] / 100
if "u10"   in gm.columns and "v10" in gm.columns:
    gm["wspd_kt"] = np.sqrt(gm["u10"]**2 + gm["v10"]**2) * 1.944
    gm["wdir"]    = (270 - np.degrees(np.arctan2(gm["v10"], gm["u10"]))) % 360
if "fg10"  in gm.columns: gm["fg10_kt"] = gm["fg10"] * 1.944
if "d2m"   in gm.columns: gm["d2m_c"]   = gm["d2m"] - 273.15
if "tp"    in gm.columns: gm["tp_mm"]   = gm["tp"].diff().clip(lower=0) * 1000

# New tropical columns from extended GEFS download
if "tcdc"  in gm.columns: gm["tcc_pct"] = gm["tcdc"] if gm["tcdc"].max() > 1 else gm["tcdc"] * 100
if "pwat"  in gm.columns and "tcwv" not in gm.columns: gm["tcwv"]    = gm["pwat"]
if "ulwrf" in gm.columns: gm["olr_wm2"] = gm["ulwrf"]
if "shtfl" in gm.columns: gm["shf"]      = gm["shtfl"]
if "lhtfl" in gm.columns: gm["lhf"]      = gm["lhtfl"]
if "dswrf" in gm.columns: gm["ssrd_wm2"] = gm["dswrf"]
if "dlwrf" in gm.columns: gm["strd_wm2"] = gm["dlwrf"]
# GEFS mean/spread has no lcc/mcc/hcc, blh, skt, cin — will be NaN in canonical CSV

export = pd.DataFrame(
    {col: gm[col].values if col in gm.columns else [float("nan")] * len(gm)
     for col in CANONICAL_COLS},
    index=gm.index,
)
export.insert(0, "model", "noaa_gefs")
export.index.name = "valid_time"

csv_path = RUN_DIR / "canonical_sfc.csv"
export.to_csv(csv_path, float_format="%.4f")
non_nan = export.notna().sum().sum()
print(f"Saved: {csv_path.name}  ({len(export)} steps, {non_nan} non-NaN values)")
print(export[["t2m_c", "msl_hpa", "wspd_kt", "fg10_kt", "cape", "olr_wm2", "lhf"]].round(2).to_string())